In [ ]:
# Performs analysis of average variograms for 4 seasons 
# within each cluster to derive optimal parameters for kriging
#  Fix 1| 26 April | using ERA5L data as input for kriging instead of insitu data

# Workflow steps:
# 1. Create single seasonal raster from all daily images
# 2. Dissolve pixels to points and transform projections to UTM 45
# * this step onwards require cluster + season subsetting 
# 3. Detect linear trend and create residuals
# 4. Apply nscore transform (check how it works out before and after)
# 5. Isotropy check (check directional variogram; only flag anisotropy and 
# do not actually extend it to the workflow since we are assuming isotropy within cluster)
# 6. Create isotropy assumed experimental variogram with WLS and tweak parameters 
# 7. Perfrom LOOCV to validate parameter choices and select the best visually + metrically
# 8. Store best parameters in a csv

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from pyproj import Proj
import geostatspy.geostats as geostats
import geostatspy.GSLIB as GSLIB

import os
import glob
import rasterio
import rioxarray as rxr
import xarray as xr
from rasterio.enums import Resampling
import dask
import sys

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from scipy.stats import shapiro
from sklearn.preprocessing import PolynomialFeatures
from scipy.optimize import curve_fit

#Apply variables
season = 'winter'
cluster_id = 3         #pick among 0 to 3
r2_threshold = 0.05    #trend removal 

#define seasons
seasons = {
        'winter': [12, 1, 2],
        'pre_monsoon': [3, 4, 5],
        'monsoon': [6, 7, 8, 9],
        'post_monsoon': [10, 11]
    }

temp_file_path = r"..\Data\Processed\Variogram_analysis\Temp"
os.makedirs(temp_file_path, exist_ok = True)   

In [ ]:
##1. Create single seasonal raster from all daily images 
#uncomment and run if required

# era5L_files_path = r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\ERA5L_prec"
# era5L_files = sorted(glob.glob(os.path.join(era5L_files_path, "*.tif")))
# era5L_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in era5L_files]
# era5L_stack = xr.concat([rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}, masked = True) for f in era5L_files], 
#                 dim=pd.Index(era5L_dates, name='time'), join = 'exact')
# era5L_stack = era5L_stack.squeeze('band', drop = True)


# for season, months in seasons.items():
#     mask = era5L_stack.time.dt.month.isin(months)
#     stack = era5L_stack.isel(time=mask)

#     if season == 'winter':
#         time_index = pd.DatetimeIndex(stack.time.values)
#         season_year = np.where(
#             time_index.month == 12,
#             time_index.year + 1,
#             time_index.year
#         )
#         stack = stack.assign_coords(season_year=('time', season_year))
#     else:
#         season_year = pd.DatetimeIndex(stack.time.values).year
#         stack = stack.assign_coords(season_year=('time', season_year))

#     annual_sums = stack.groupby('season_year').sum(dim='time', skipna=True)
#     climatology  = annual_sums.mean(dim='season_year', skipna=True)
#     climatology.rio.to_raster(os.path.join(temp_file_path, f"{season}_clim.tif"))

# del era5L_stack, stack, annual_sums, climatology


In [ ]:
# We'll implement seasonal and cluster filter onwards this during execution
## 2. Dissolve pixels to points and transform projections to UTM 45
climatology_rxr = rxr.open_rasterio(os.path.join(temp_file_path, f"{season}_clim.tif"))
climatology_rxr = climatology_rxr.squeeze('band', drop = True)
cluster_rxr = rxr.open_rasterio(r"..\Data\Processed\Spatial_clustering\Precipitation_clusters.tif")
cluster_coarse = cluster_rxr.rio.reproject_match(
        climatology_rxr,
        resampling=Resampling.mode  
    )
cluster_mask = (cluster_coarse == cluster_id)
era5l_cluster = climatology_rxr.where(cluster_mask)
climatology_df = era5l_cluster.to_dataframe(name = "prec").reset_index()
climatology_df = climatology_df.drop(columns= ['spatial_ref','band']).dropna().reset_index()

myProj = Proj("+proj=utm +zone=45 +ellps=WGS84 +datum=WGS84 +units=m +no_defs")
x, y = myProj(climatology_df['x'].values, climatology_df['y'].values)
climatology_df['x_km'] = x / 1000.0
climatology_df['y_km'] = y / 1000.0
climatology_df = climatology_df[['x_km','y_km','prec']]

In [ ]:
climatology_df[['prec']].min()

In [ ]:
## 3. Detect linear trend and create residuals
X_coord = climatology_df[['x_km', 'y_km']].values
y_prec = climatology_df['prec'].values

# a pipeline is built that
# first linear features are expanded to polynomial
poly = PolynomialFeatures(degree = 2, include_bias = False)
# then a linear regression is fitted on the expanded features
model = Pipeline([
    ('poly_features', poly),
    ('linear_regression', LinearRegression())
])

model = model.fit(X_coord, y_prec)
predictions = model.predict(X_coord)
r2 = model.score(X_coord, y_prec)

# Extract internals for printing
lm        = model.named_steps['linear_regression']
coef      = lm.coef_          
intercept = lm.intercept_
feat_names = poly.get_feature_names_out(['x_km', 'y_km'])


print(f"Trend Surface Fit (R²): {r2:.4f}")
print(f"\n2nd-Degree Polynomial Trend Surface:")
print(f"  z = ", end="")
terms = [f"{c:.4f}*{n}" for c, n in zip(coef, feat_names)]
print(" + ".join(terms) + f" + {intercept:.4f}")


# lm = LinearRegression().fit(X_coord, y_prec)
# predictions = lm.predict(X_coord)
# r2 = lm.score(X_coord, y_prec)

In [ ]:
## 4. Apply nscore transform (check how it works out before and after)

if r2 >= r2_threshold:
    residuals = y_prec - predictions
    print("Residuals calculated")    
else:
    residuals = y_prec - y_prec.mean()
    print("Residuals are not calculated. Precipitation values are used directly.")

# Check normality before nscore transformation
shapiro_stat, shapiro_pval = shapiro(residuals)
print(f"Shapiro test for normality, stat: {shapiro_stat}, p-value: {shapiro_pval}")

# N-score transformation is used regardless for uniformity
ns_data, t_values, t_nscore = geostats.nscore(
    pd.DataFrame({'val': residuals}), 'val'
)

lookup_df = pd.DataFrame({
    'original_value': t_values,  
    'nscore_value':   t_nscore   
})

lookup_path = os.path.join(
    temp_file_path,
    f"nscore_lookup_{season}_cluster{cluster_id}.csv"
)
lookup_df.to_csv(lookup_path, index=False)

In [ ]:
## 5. Isotropy check (check directional variogram; only flag anisotropy) - sonnet 4.6 generated (unchecked)
#here we assume arbritray gamv parameters out of convention to check if semivariance plots on 4 major directions look roughly the same within the cluster

# ── Parameters ────────────────────────────────────────────────────────────
# Use the same lag geometry for all directions so plots are comparable
lag_dist  = 22.0    # km — approx 2x ERA5-L spacing
lag_tol   = 11.0    # km — half of lag_dist
n_lag     = 15      # bins — covers ~330 km max lag
bandh     = 9999 # no bandwidth restriction (omnidirectional within each azimuth cone)
tmin, tmax = -9999, 9999

# Four cardinal directions: azimuth (degrees from N), tolerance (±22.5 covers 45° cone)
directions = [
    {'azm': 0,   'atol': 22.5, 'label': '0° (N-S)'},
    {'azm': 45,  'atol': 22.5, 'label': '45° (NE-SW)'},
    {'azm': 90,  'atol': 22.5, 'label': '90° (E-W)'},
    {'azm': 135, 'atol': 22.5, 'label': '135° (NW-SE)'},
]

# ── Attach nscore values to DataFrame for gamv ────────────────────────────
climatology_df['ns'] = ns_data

# ── Compute directional variograms ────────────────────────────────────────
dir_variograms = {}

for d in directions:
    lag_h, gamma_h, npairs_h = geostats.gamv(
        climatology_df, 'x_km', 'y_km', 'ns',
        tmin, tmax,
        lag_dist, lag_tol, n_lag,
        d['azm'], d['atol'], bandh,
        isill=0    # raw semivariance — do not normalise, sill should be ≈1.0 for nscore
    )
    dir_variograms[d['label']] = {
        'lag':    lag_h,
        'gamma':  gamma_h,
        'npairs': npairs_h
    }

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['steelblue', 'darkorange', 'forestgreen', 'crimson']

# Left panel: all four directions on one plot — main isotropy check
for (label, vdata), color in zip(dir_variograms.items(), colors):
    valid = np.isfinite(vdata['gamma']) & (vdata['npairs'] > 5)
    axes[0].plot(
        vdata['lag'][valid], vdata['gamma'][valid],
        'o-', color=color, label=label, linewidth=1.5, markersize=5
    )

axes[0].axhline(1.0, color='gray', linestyle='--', linewidth=1, label='Theoretical sill (nscore)')
axes[0].set_xlabel('Lag distance (km)')
axes[0].set_ylabel('Semivariance')
axes[0].set_title(f'Directional variograms — {season}, cluster {cluster_id}')
axes[0].legend(fontsize=9)
axes[0].set_xlim(left=0)
axes[0].set_ylim(bottom=0)
axes[0].grid(True, alpha=0.3)

# Right panel: pair counts per lag per direction — reliability check
for (label, vdata), color in zip(dir_variograms.items(), colors):
    valid = np.isfinite(vdata['npairs'])
    axes[1].plot(
        vdata['lag'][valid], vdata['npairs'][valid],
        'o-', color=color, label=label, linewidth=1.5, markersize=5
    )

axes[1].axhline(30, color='black', linestyle=':', linewidth=1.2, label='Min reliable pairs (30)')
axes[1].set_xlabel('Lag distance (km)')
axes[1].set_ylabel('Number of pairs')
axes[1].set_title('Pair counts per directional lag bin')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle(
    f'Isotropy check | Season: {season} | Cluster: {cluster_id} | Sonnet 4.6 generated',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

# ── Isotropy flag ─────────────────────────────────────────────────────────
# Automated heuristic: compare range at which each direction reaches 80% of sill (≈0.8)
# If the max/min ratio of those ranges exceeds 2.0, flag potential anisotropy
iso_ranges = {}
sill_target = 0.8  # 80% of theoretical sill

for label, vdata in dir_variograms.items():
    valid = np.isfinite(vdata['gamma']) & (vdata['npairs'] > 5)
    lag_v   = vdata['lag'][valid]
    gamma_v = vdata['gamma'][valid]
    # Find first lag where semivariance exceeds sill_target
    crossed = lag_v[gamma_v >= sill_target]
    iso_ranges[label] = float(crossed[0]) if len(crossed) > 0 else np.nan

print("\n── Isotropy diagnostic ──────────────────────────────────")
for label, r in iso_ranges.items():
    print(f"  {label}: reaches 80% sill at {r:.1f} km" if not np.isnan(r) else f"  {label}: never reaches 80% sill within max lag")

valid_ranges = [r for r in iso_ranges.values() if not np.isnan(r)]
if len(valid_ranges) >= 2:
    anis_ratio = min(valid_ranges) / max(valid_ranges)
    isotropic_flag = anis_ratio >= 0.5
    print(f"\n  Range ratio (max/min): {anis_ratio:.2f}")
    print(f"  Isotropy assumption: {'✓ DEFENSIBLE' if isotropic_flag else '⚠ POTENTIAL ANISOTROPY — log in notes'}")
else:
    isotropic_flag = True
    print("  Insufficient directional data to assess — defaulting to isotropic")

print(f"\n  → isotropic = {isotropic_flag}  (stored for output CSV)")
 


In [ ]:
## 6. Create isotropy assumed experimental variogram with WLS and tweak parameters

# check anisotropy ratio 
major_label = max(iso_ranges, key=lambda k: iso_ranges[k] if not np.isnan(iso_ranges[k]) else -1)
azm_major = next(d['azm'] for d in directions if d['label'] == major_label)
azm_minor = (azm_major + 90) % 180


# variogram is fitted omnidirectionally or on the major axis based on the anisotropy ratio
if anis_ratio < 0.8:
    # fit from major-axis cone to avoid contamination from noisy minor-axis pairs
    fit_azm  = azm_major
    fit_atol = 45
    directions_used = f'{azm_major}°±45°'
else:
    fit_azm  = 0
    fit_atol = 90        # full omnidirectional
    directions_used = 'omnidirectional'

lag_exp, gamma_exp, n_pairs = geostats.gamv(
    climatology_df, 'x_km', 'y_km', 'ns',
    tmin=-9999, tmax=9999,
    xlag=lag_dist, xltol=lag_tol, nlag=n_lag,
    azm=fit_azm, atol=fit_atol, bandwh=9999,
    isill=0
)

In [ ]:
#guards for gamv parameters
valid_bins = (n_pairs >= 30) & (lag_exp <= lag_cap)

lag_fit    = lag_exp[valid_bins]
gamma_fit  = gamma_exp[valid_bins]
npairs_fit = n_pairs[valid_bins]
n_pairs_min = int(npairs_fit.min()) if len(npairs_fit) > 0 else 0

# Hard stop only if almost nothing survives
if len(lag_fit) < 4:
    raise ValueError(f"Only {len(lag_fit)} valid bins after filtering — cannot fit variogram")

In [ ]:
# wls weight computation - iterative cressie - sonnet 4.6 generated + checked

# define model functions - exponential, gaussian and spherical  
def spherical(h, nugget, partial_sill, range_):
    h = np.asarray(h, dtype=float)
    sph = np.where(
        h <= range_,
        nugget + partial_sill * (1.5*(h/range_) - 0.5*(h/range_)**3),
        nugget + partial_sill
    )
    return np.where(h == 0, nugget, sph)

def exponential(h, nugget, partial_sill, range_):
    h = np.asarray(h, dtype=float)
    return np.where(h == 0, nugget,
                    nugget + partial_sill * (1 - np.exp(-h / (range_/3))))

def gaussian(h, nugget, partial_sill, range_):
    h = np.asarray(h, dtype=float)
    return np.where(h == 0, nugget,
                    nugget + partial_sill * (1 - np.exp(-(h**2) / (range_/3)**2)))

model_functions = {
    'spherical': spherical,
    'exponential': exponential,
    'gaussian': gaussian
}

# initialize parameters and bounds
range_init = iso_ranges.get(major_label, lag_fit.mean())
p0           = [0.05, 0.90, range_init]
cluster_extent = pdist(climatology_df[['x_km','y_km']].values).max()
bounds       = ([0, 0, lag_dist],
                [1.0, 1.0, 2 * cluster_extent])

n_bins = len(lag_fit)

def fit_model(func, lag_fit, gamma_fit, sigma, p0, bounds):
    popt, _ = curve_fit(
        func, lag_fit, gamma_fit,
        p0=p0, sigma=sigma, absolute_sigma=True,
        bounds=bounds, maxfev=10000
    )
    return popt

def iterative_wls(func, lag_fit, gamma_fit, npairs_fit, p0, bounds, n_iter=3, conv_tol=0.01):
    """
    Cressie (1985) iterative WLS for one variogram model function.

    Returns
    -------
    popt_final : array  [nugget, partial_sill, range]
    sigma_final: array  per-bin sigma used in the last fit
    converged  : bool
    history    : list of popt arrays per iteration (for diagnostics)
    """

    # ── Iteration 0: warm-start weights (n_pairs / h²) ───────────────────
    warm_weights = npairs_fit / np.maximum(lag_fit**2, 1e-6)
    warm_weights /= warm_weights.sum() / n_bins          # normalise
    sigma_warm   = 1.0 / np.sqrt(np.maximum(warm_weights, 1e-12))

    try:
        popt = fit_model(func, lag_fit, gamma_fit, sigma_warm, p0, bounds)
    except RuntimeError:
        raise RuntimeError("Warm-start fit failed — check p0 and bounds")

    history   = [popt.copy()]
    converged = False

    # ── Iterations 1–n_iter: Cressie weights ─────────────────────────────
    for i in range(1, n_iter + 1):
        gamma_model = func(lag_fit, *popt)

        # Cressie (1985): w(h) = N(h) / γ_model(h)²
        denom   = np.maximum(gamma_model, 1e-6) ** 2
        weights = npairs_fit / denom
        weights = weights / weights.sum() * n_bins       # normalise to n_bins
        sigma   = 1.0 / np.sqrt(np.maximum(weights, 1e-12))

        try:
            popt_new = fit_model(func, lag_fit, gamma_fit, sigma, popt, bounds)
        except RuntimeError:
            print(f"  ⚠  Iteration {i} failed to converge — keeping previous popt")
            break

        # ── Convergence check ─────────────────────────────────────────────
        rel_change = np.abs((popt_new - popt) / np.maximum(np.abs(popt), 1e-9))
        history.append(popt_new.copy())
        popt = popt_new

        if rel_change.max() < conv_tol:
            converged = True
            print(f"  ✓  Converged at iteration {i} (max param change "
                  f"{rel_change.max()*100:.3f}%)")
            break

    if not converged:
        print(f"  ⚠  Did not converge within {n_iter} iterations "
              f"(max param change {rel_change.max()*100:.2f}%)")

    return popt, sigma, converged, history


# ── Run for all three models ──────────────────────────────────────────────
wls_results = {}

for model_name, func in model_functions.items():
    try:
        popt, sigma_final, converged, history = iterative_wls(
            func, lag_fit, gamma_fit, npairs_fit, p0, bounds
        )
        nugget, partial_sill, range_km = popt
        gamma_model = func(lag_fit, *popt)

        # ── Diagnostics ──────────────────────────────────────────────────
        ss_res = np.sum(((gamma_fit - gamma_model) / sigma_final)**2)
        ss_tot = np.sum(((gamma_fit - gamma_fit.mean()) / sigma_final)**2)
        wls_r2 = 1 - ss_res / ss_tot if ss_tot > 1e-12 else np.nan
        wls_rmse = np.sqrt(np.mean((gamma_fit - gamma_model)**2))

        sill_flag = abs(nugget + partial_sill - 1.0) > 0.2

        wls_results[model_name] = {
            'func':         func,
            'popt':         popt,
            'sigma_final':  sigma_final,
            'converged':    converged,
            'history':      history,
            'nugget':       nugget,
            'partial_sill': partial_sill,
            'total_sill':   nugget + partial_sill,
            'range_km':     range_km,
            'wls_r2':       wls_r2,
            'wls_rmse':     wls_rmse,
            'sill_flag':    sill_flag,
            'disqualified': False,
            'disq_reason':  None
        }

        print(f"\n{model_name.capitalize()}")
        print(f"  nugget={nugget:.4f}  partial_sill={partial_sill:.4f}  "
              f"total_sill={nugget+partial_sill:.4f}  range={range_km:.2f} km")
        print(f"  WLS R²={wls_r2:.4f}  RMSE={wls_rmse:.4f}  "
              f"converged={converged}"
              + ("  ⚠ SILL FLAG" if sill_flag else ""))

    except RuntimeError as e:
        print(f"\n{model_name.capitalize()}: FAILED — {e}")
        wls_results[model_name] = {'disqualified': True, 'disq_reason': str(e)}

In [ ]:
# 7. Perfrom LOOCV to validate parameter choices and select the best visually + metrically

# ── 7a. Parameter override block — tweak here before plotting/LOOCV ──────
model_params = {}
for model_name, res in wls_results.items():
    if res.get('disqualified'):
        continue
    model_params[model_name] = {
        'nugget':       res['nugget'],
        'partial_sill': res['partial_sill'],
        'range_km':     res['range_km'],
    }

# Uncomment and edit any line below to override fitted values manually
# model_params['spherical']['nugget']       = 0.05
# model_params['spherical']['partial_sill'] = 0.88
# model_params['spherical']['range_km']     = 145.0
# model_params['exponential']['nugget']     = 0.05
# model_params['exponential']['partial_sill']= 0.90
# model_params['exponential']['range_km']   = 180.0
# model_params['gaussian']['nugget']        = 0.02
# model_params['gaussian']['partial_sill']  = 0.92
# model_params['gaussian']['range_km']      = 120.0

MODEL_IT = {'spherical': 1, 'exponential': 2, 'gaussian': 3}
COLORS   = {'spherical': 'steelblue', 'exponential': 'darkorange', 'gaussian': 'forestgreen'}

In [ ]:
# ── 7b. Visual comparison of fitted models ────────────────────────────────
h_plot = np.linspace(0, lag_fit.max() * 1.1, 300)

fig, ax = plt.subplots(figsize=(10, 5))

# Experimental points — size proportional to n_pairs reliability
scatter_sizes = (npairs_fit / npairs_fit.max()) * 120 + 20
ax.scatter(lag_fit, gamma_fit, s=scatter_sizes, color='black',
           zorder=5, label='Experimental (size ∝ n_pairs)', alpha=0.85)

# n_pairs annotation per bin
for h, g, n in zip(lag_fit, gamma_fit, npairs_fit):
    ax.annotate(str(int(n)), (h, g),
                textcoords='offset points', xytext=(0, 7),
                ha='center', fontsize=7, color='gray')

# Model curves
for model_name, params in model_params.items():
    func        = model_functions[model_name]
    gamma_curve = func(h_plot, params['nugget'],
                       params['partial_sill'], params['range_km'])
    wls_r2      = wls_results[model_name]['wls_r2']
    label = (f"{model_name.capitalize()} | "
             f"nug={params['nugget']:.3f}  cc={params['partial_sill']:.3f}  "
             f"range={params['range_km']:.1f} km | WLS R²={wls_r2:.3f}")
    ax.plot(h_plot, gamma_curve,
            color=COLORS[model_name], linewidth=2.2, label=label)

ax.axhline(1.0, color='gray', linestyle='--', linewidth=1,
           label='Theoretical sill (nscore ≈ 1.0)')
ax.set_xlabel('Lag distance (km)')
ax.set_ylabel('Semivariance')
ax.set_title(f'Fitted variogram models | Season: {season} | Cluster: {cluster_id}')
ax.legend(fontsize=8)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7c. LOOCV using kb2d 1-point grid ────────────────────────────────────
# NOTE: isotropy assumed within cluster (per Step 5 decision)
# so hmaj1 = hmin1 = range_km for all models

coords   = climatology_df[['x_km', 'y_km']].values
ns_vals  = climatology_df['ns'].values
n_points = len(climatology_df)

# radius must always encompass all remaining training points
loocv_radius = pdist(coords).max() * 1.1

loocv_results = {}

for model_name, params in model_params.items():

    vario = GSLIB.make_variogram(
        nug=params['nugget'],
        nst=1,
        it1=MODEL_IT[model_name],
        cc1=params['partial_sill'],
        azi1=azm_major,
        hmaj1=params['range_km'],
        hmin1=params['range_km']     # isotropic assumption
    )

    z_pred = np.full(n_points, np.nan)
    z_kvar = np.full(n_points, np.nan)

    for i in range(n_points):
        loo_df = climatology_df.drop(index=i).reset_index(drop=True)
        x0, y0 = coords[i]

        kval, kvar = geostats.kb2d(
            loo_df, 'x_km', 'y_km', 'ns',
            tmin=-9999, tmax=9999,
            nx=1, xmn=x0, xsiz=0.001,
            ny=1, ymn=y0, ysiz=0.001,
            nxdis=1, nydis=1,
            ndmin=1, ndmax=n_points,
            radius=loocv_radius,
            ktype=0, skmean=0.0,
            vario=vario
        )
        z_pred[i] = kval.flatten()[0]
        z_kvar[i] = kvar.flatten()[0]

    # ── Metrics ──────────────────────────────────────────────────────────
    # valid mask: prediction exists and kriging variance is positive
    valid    = np.isfinite(z_pred) & np.isfinite(z_kvar) & (z_kvar > 1e-10)
    n_failed = int((~valid).sum())

    actual = ns_vals[valid]
    pred   = z_pred[valid]
    kvar_v = z_kvar[valid]

    std_errors = (pred - actual) / np.sqrt(kvar_v)

    loocv_mse  = float(std_errors.mean())           # bias — target ≈ 0
    loocv_msse = float((std_errors ** 2).mean())    # variance calibration — target ≈ 1.0

    ss_res     = float(np.sum((actual - pred) ** 2))
    ss_tot     = float(np.sum((actual - actual.mean()) ** 2))
    loocv_r2   = 1.0 - ss_res / ss_tot if ss_tot > 1e-12 else np.nan
    loocv_mae  = float(np.mean(np.abs(actual - pred)))

    loocv_results[model_name] = {
        'loocv_mse':   loocv_mse,
        'loocv_msse':  loocv_msse,
        'loocv_r2':    loocv_r2,
        'loocv_mae':   loocv_mae,
        'n_failed':    n_failed,
    }



In [ ]:

print(f"\n── LOOCV Results | Season: {season} | Cluster: {cluster_id} ──────────────────")
print(f"{'Model':<14} {'R²':>8} {'MAE':>8} {'MSE':>8} {'MSSE':>8} {'Failed':>8}")
print("─" * 62)
for model_name, m in loocv_results.items():
    print(f"{model_name.capitalize():<14} "
          f"{m['loocv_r2']:>8.4f} "
          f"{m['loocv_mae']:>8.4f} "
          f"{m['loocv_mse']:>+8.4f} "
          f"{m['loocv_msse']:>8.4f} "
          f"{m['n_failed']:>8}")
print("─" * 62)
print(f"{'Targets':<14} {'higher':>8} {'lower':>8} {'≈ 0.0':>8} {'≈ 1.0':>8}")